In [1]:
import pandas as pd
import numpy as np
import random

In [2]:
original_df = pd.read_csv("../data/raw/positive_rep_virus_dataset.csv")

In [3]:
original_df

,taxid,virus_name,sDBM_pos,sDBM,sDBM_sim_gap,ori_seq,ori_align_score,iteration_name,rep_amino_acid_seq
0,59817,Faba bean necrotic yellows virus,"(63, 71)",GAHFEKRRG,33.0,TAGTATTAC,138.5,FBNYY_iter1_V_A,MARQVICWCFTLNNPLSPLSLHDSMKYLVYQTEQGEAGNIHFQGYI...
1,262588,Cardamom bushy dwarf virus,"(63, 71)",GAHLEKRKG,33.1,TATTATTAC,138.2,FBNYY_iter1_V_A,MARYVVCWMFTINNPSSLPVMRDEIKYMVYQVERGQEGTRHVQGYV...
2,1819678,Cucurbit yellow mosaic alphasatellite,"(66, 74)",GTHIEACKG,36.2,GATTTGTAC,129.9,FBNYY_iter1_V_A,MSHSRNWCFTIFNYVLPLFTTLPEWANYLIYQEEECPTTKKRHIQG...
3,1260769,Colecusatellite agerati,"(73, 81)",KPHLEKQRA,22.3,GACTGCTAC,130.0,FBNYY_iter1_V_A,MAAIKGQWWCFTVFFLTATAPDLVPLFENTHVSYACWQEEESPTTK...
4,2169722,Banana bunchy top alphasatellite 2,"(70, 78)",RAHWEKARG,40.3,TAGTATTAC,138.5,FBNYY_iter1_V_A,MSASRWTFTLHYSDATERGKFLATLKEEDVHYAVVGDETAPNTGRK...
...,...,...,...,...,...,...,...,...,...
279,222450,Chayote yellow mosaic virus,"(88, 96)",HPNVQGAKS,37.7,TTATACTAC,132.4,SGC1_iter2_V_R,MAPPKRFQINAKNYFLTYPQCSLTKEEALSQLRSLRTPTNQKYIKV...
280,223332,Tobacco leaf curl Japan virus,"(88, 96)",HPNIQGAKS,37.9,TAATATTAC,139.3,SGC1_iter2_V_R,MAPPKRFQINAKNYFLTYPHCSLTKEEALSQIQALDTITNKLFIRI...
281,486809,Ramie mosaic virus,"(88, 96)",HPNIQGAKS,37.4,TAATATTAC,139.3,SGC1_iter2_V_R,MAPPRRFTINAKNYFLTYPQCPLSKEEALSQLLNLSTPTNKKFIKI...
282,390437,Kudzu mosaic virus,"(88, 96)",HPNIQGAKS,37.0,TTATCCTAC,130.6,SGC1_iter2_V_R,MSRPKGFRVNAKNFFLTYPRCSLSKEAALEQLQNIQTNVNKKFIRV...


In [4]:
# Remove NaN ori
original_df.dropna(
    subset=['ori_seq', 'sDBM_pos', 'rep_amino_acid_seq'],
    inplace=True
)
 #inplace=True avoids reassignment of dataset

# Drop entries where sDBM confidence < 10
df_sDBM_cleaned = original_df[original_df['sDBM_sim_gap'] >= 10]

# Drop entries where ori score < 130
df = df_sDBM_cleaned[df_sDBM_cleaned['ori_align_score'] >= 130]

len(df)


238

In [16]:
# # weed out any bacteria

# #ASSUMPTION: if virus, it has the 'virus' somewhere in the name
# def is_Virus_organism(organism_name):

#     return "virus" in organism_name.lower()

#     # if "virus" in organism_name.lower():
#     #     return True

In [119]:
# print(len(df))

# # Drop suspected bacteria entries
# mask = df['virus_name'].apply(is_Virus_organism)

# print(mask)

# false_elements = df.index[~mask]

# print(false_elements)

# df = df[mask]


# len(df)

In [120]:
# def get_suspected_bacteria(df, rows):

#     suspected = []

#     for row in rows:

#         organism_name = original_df.iat[row, 1] 

#         suspected.append(organism_name)

#     print(suspected)

In [121]:
# get_suspected_bacteria(df, false_elements)

In [5]:
negative_df = df.copy()

In [6]:
def shuffle_until_no_match(max_attempts=10000):

    minMatches = np.inf
    best_shuffle = None
    best_shuffle_same_ori_rows = None

    for _ in range(max_attempts):
        negative_df['ori_seq'] = np.random.permutation(negative_df['ori_seq'])

        boolarr = negative_df['ori_seq'].values == df['ori_seq'].values


        if not np.any(boolarr):
            print("Perfect shuffle")
            return

        if boolarr.sum() < minMatches:
            minMatches = boolarr.sum()
            best_shuffle_same_ori_rows = np.where(boolarr)[0]
            best_shuffle = negative_df.copy()

    
    print(best_shuffle_same_ori_rows)


    print("Failed")

    print(minMatches)

    return best_shuffle, best_shuffle_same_ori_rows, minMatches

In [55]:
#for misses, just randomly mutate the ori

In [7]:
best_negative_df, indices, minMatches = shuffle_until_no_match()

[21 34 57 92 95]
Failed
5


In [8]:
best_negative_df

,taxid,virus_name,sDBM_pos,sDBM,sDBM_sim_gap,ori_seq,ori_align_score,iteration_name,rep_amino_acid_seq
0,59817,Faba bean necrotic yellows virus,"(63, 71)",GAHFEKRRG,33.0,CAGTATTAC,138.5,FBNYY_iter1_V_A,MARQVICWCFTLNNPLSPLSLHDSMKYLVYQTEQGEAGNIHFQGYI...
1,262588,Cardamom bushy dwarf virus,"(63, 71)",GAHLEKRKG,33.1,TAATATTAC,138.2,FBNYY_iter1_V_A,MARYVVCWMFTINNPSSLPVMRDEIKYMVYQVERGQEGTRHVQGYV...
3,1260769,Colecusatellite agerati,"(73, 81)",KPHLEKQRA,22.3,GAATCATAC,130.0,FBNYY_iter1_V_A,MAAIKGQWWCFTVFFLTATAPDLVPLFENTHVSYACWQEEESPTTK...
4,2169722,Banana bunchy top alphasatellite 2,"(70, 78)",RAHWEKARG,40.3,ACGTTTTAC,138.5,FBNYY_iter1_V_A,MSASRWTFTLHYSDATERGKFLATLKEEDVHYAVVGDETAPNTGRK...
6,1685737,Lake Sarah-associated circular virus-11,"(74, 82)",RAHFECAKG,40.2,TAGTGTTAC,137.7,FBNYY_iter1_V_A,MAPRGNAAKRWTFTINNWTLADYDKVLEFTTKFEPVFLICGKERGE...
...,...,...,...,...,...,...,...,...,...
279,222450,Chayote yellow mosaic virus,"(88, 96)",HPNVQGAKS,37.7,CAATGATAC,132.4,SGC1_iter2_V_R,MAPPKRFQINAKNYFLTYPQCSLTKEEALSQLRSLRTPTNQKYIKV...
280,223332,Tobacco leaf curl Japan virus,"(88, 96)",HPNIQGAKS,37.9,AAATGTTAC,139.3,SGC1_iter2_V_R,MAPPKRFQINAKNYFLTYPHCSLTKEEALSQIQALDTITNKLFIRI...
281,486809,Ramie mosaic virus,"(88, 96)",HPNIQGAKS,37.4,TAGTATTAC,139.3,SGC1_iter2_V_R,MAPPRRFTINAKNYFLTYPQCPLSKEEALSQLLNLSTPTNKKFIKI...
282,390437,Kudzu mosaic virus,"(88, 96)",HPNIQGAKS,37.0,TAGTATTAC,130.6,SGC1_iter2_V_R,MSRPKGFRVNAKNFFLTYPRCSLSKEAALEQLQNIQTNVNKKFIRV...


In [9]:
# in place modification

def mutate_ori_dataset(df, rows):

    bases = ['A', 'C', 'T', 'G']

    for row in rows:

        random_pos =random.randint(0, 6)
        random_base = random.choice(bases)

        same_ori = df.iat[row.item(), 5] 

        # avoid it from being replaced by the exact same base (by chance)
        while(random_base == same_ori[random_pos]):
            random_base = random.choice(bases)


        mutated_ori = same_ori[:random_pos] + random_base + same_ori[random_pos + 1:]
        print(same_ori)
        print(mutated_ori)

        df.iat[row.item(), 5] = mutated_ori

        

In [10]:
mutate_ori_dataset(best_negative_df,indices)

TAGTATTAC
TGGTATTAC
TAGTATTAC
TGGTATTAC
TAGTATTAC
TAGCATTAC
TAGTATTAC
TAGCATTAC
TAGTATTAC
TAGGATTAC


In [11]:
#check if mutation worked

# np.any must be false -> no ori matches in positive and negative dataset
if(np.any(best_negative_df['ori_seq'].values == df['ori_seq'].values)):
    print("Negative dataset has contradictions")
else:
    print("Success! All ori are different!")


Success! All ori are different!


In [12]:
df['match'] = "Yes"
best_negative_df['match'] = "No"

print(len(df))
print(len(best_negative_df))

final_df = pd.concat([df, best_negative_df], ignore_index=True)

238
238


In [13]:
print(len(final_df))
final_df

476


,taxid,virus_name,sDBM_pos,sDBM,sDBM_sim_gap,ori_seq,ori_align_score,iteration_name,rep_amino_acid_seq,match
0,59817,Faba bean necrotic yellows virus,"(63, 71)",GAHFEKRRG,33.0,TAGTATTAC,138.5,FBNYY_iter1_V_A,MARQVICWCFTLNNPLSPLSLHDSMKYLVYQTEQGEAGNIHFQGYI...,Yes
1,262588,Cardamom bushy dwarf virus,"(63, 71)",GAHLEKRKG,33.1,TATTATTAC,138.2,FBNYY_iter1_V_A,MARYVVCWMFTINNPSSLPVMRDEIKYMVYQVERGQEGTRHVQGYV...,Yes
2,1260769,Colecusatellite agerati,"(73, 81)",KPHLEKQRA,22.3,GACTGCTAC,130.0,FBNYY_iter1_V_A,MAAIKGQWWCFTVFFLTATAPDLVPLFENTHVSYACWQEEESPTTK...,Yes
3,2169722,Banana bunchy top alphasatellite 2,"(70, 78)",RAHWEKARG,40.3,TAGTATTAC,138.5,FBNYY_iter1_V_A,MSASRWTFTLHYSDATERGKFLATLKEEDVHYAVVGDETAPNTGRK...,Yes
4,1685737,Lake Sarah-associated circular virus-11,"(74, 82)",RAHFECAKG,40.2,CAGTATTAC,137.7,FBNYY_iter1_V_A,MAPRGNAAKRWTFTINNWTLADYDKVLEFTTKFEPVFLICGKERGE...,Yes
...,...,...,...,...,...,...,...,...,...,...
471,222450,Chayote yellow mosaic virus,"(88, 96)",HPNVQGAKS,37.7,CAATGATAC,132.4,SGC1_iter2_V_R,MAPPKRFQINAKNYFLTYPQCSLTKEEALSQLRSLRTPTNQKYIKV...,No
472,223332,Tobacco leaf curl Japan virus,"(88, 96)",HPNIQGAKS,37.9,AAATGTTAC,139.3,SGC1_iter2_V_R,MAPPKRFQINAKNYFLTYPHCSLTKEEALSQIQALDTITNKLFIRI...,No
473,486809,Ramie mosaic virus,"(88, 96)",HPNIQGAKS,37.4,TAGTATTAC,139.3,SGC1_iter2_V_R,MAPPRRFTINAKNYFLTYPQCPLSKEEALSQLLNLSTPTNKKFIKI...,No
474,390437,Kudzu mosaic virus,"(88, 96)",HPNIQGAKS,37.0,TAGTATTAC,130.6,SGC1_iter2_V_R,MSRPKGFRVNAKNFFLTYPRCSLSKEAALEQLQNIQTNVNKKFIRV...,No


In [30]:
final_df.to_csv('../data/processed/final_rep_virus_dataset.csv', index=False)

In [173]:
# Print entire dataset!

print(len(final_df))
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(final_df) 

476


,taxid,virus_name,sDBM_pos,sDBM,sDBM_sim_gap,ori_seq,ori_align_score,iteration_name,rep_amino_acid_seq,match
0,59817,Faba bean necrotic yellows virus,"(63, 71)",GAHFEKRRG,33.0,TAGTATTAC,138.5,FBNYY_iter1_V_A,MARQVICWCFTLNNPLSPLSLHDSMKYLVYQTEQGEAGNIHFQGYI...,Yes
1,262588,Cardamom bushy dwarf virus,"(63, 71)",GAHLEKRKG,33.1,TATTATTAC,138.2,FBNYY_iter1_V_A,MARYVVCWMFTINNPSSLPVMRDEIKYMVYQVERGQEGTRHVQGYV...,Yes
2,1260769,Colecusatellite agerati,"(73, 81)",KPHLEKQRA,22.3,GACTGCTAC,130.0,FBNYY_iter1_V_A,MAAIKGQWWCFTVFFLTATAPDLVPLFENTHVSYACWQEEESPTTK...,Yes
3,2169722,Banana bunchy top alphasatellite 2,"(70, 78)",RAHWEKARG,40.3,TAGTATTAC,138.5,FBNYY_iter1_V_A,MSASRWTFTLHYSDATERGKFLATLKEEDVHYAVVGDETAPNTGRK...,Yes
4,1685737,Lake Sarah-associated circular virus-11,"(74, 82)",RAHFECAKG,40.2,CAGTATTAC,137.7,FBNYY_iter1_V_A,MAPRGNAAKRWTFTINNWTLADYDKVLEFTTKFEPVFLICGKERGE...,Yes
5,2293298,Pimoid spider associated circular virus 2,"(66, 74)",SAHWERTKG,36.8,TAGTATTAC,138.5,FBNYY_iter1_V_A,MSCKNWCFTWNNPDLGYEALLELLSSKSSYCIFQKEQGANGTVHFQ...,Yes
6,3429808,Hudisavirus jaqis,"(73, 81)",RAHWEKTRG,37.0,GAGTATTAC,136.3,FBNYY_iter1_V_A,MSAQEGAAKRWCFTINNPTDDDKFWENSEQQEHLEYLIIQEEIGEN...,Yes
7,2800901,Marmot associated feces virus 6,"(68, 76)",KIHWEPRKG,29.4,AATTATTAC,137.5,FBNYY_iter1_V_A,MSRNRNWVFTFNNPYDIEIQMLDKILTSQADYAIYGEEVGENGTPH...,Yes
8,1234878,Dragonfly cyclicusvirus,"(67, 75)",RYHIEMRRG,37.5,TAGTATTAC,138.5,FBNYY_iter1_V_A,MGSPAIGWCFTLNNYTEIDEQVLQALDVKYLVYGHEVGEGGTPHLQ...,Yes
9,2676049,Penguin circovirus,"(73, 81)",RAHWEKARG,42.2,TAGTATTAC,138.5,FBNYY_iter1_V_A,MAPVRAAGARRWCLTLNNPTDEEVQAAKRLEPNTYHYAIIGREVGE...,Yes
